# 02 Combine drill review
Look at every OL Combine drill before deciding which drills and phases become features. Output is the review notes and decisions below, not features.

Runs on 01's clean Combine table. Writes one HTML viewer per drill to `figures/drill_review/`; open them in a browser. The viewer code lives in `shared/drill_viewer.py` and `shared/drill_viewer_template.html`.

In [ ]:
from shared.drill_viewer import align_reps, write_viewer
from shared.config import PROCESSED, FIGURES
import pandas as pd
import sys
sys.path.insert(0, '..')


CLEAN = PROCESSED / 'clean'
OUT = FIGURES / 'drill_review'

## Load OL reps
Player rows only, each rep shifted to start at (0, 0). `rep` is 1, 2, ... in time order (the raw `attempt` column is not 1 and 2).

In [2]:
cr = pd.read_parquet(PROCESSED / 'parquet' / 'nfl-big-data-bowl-2027__combine_results.parquet',
                     columns=['nfl_id', 'combine_position'])
ol_ids = cr.loc[cr['combine_position'].eq('OL'), 'nfl_id']

ct = pd.read_parquet(CLEAN / 'combine_tracking.parquet')
reps = align_reps(ct, ol_ids)
reps['nfl_id'].nunique(), reps['event_id'].nunique()

(121, 1214)

In [3]:
reps[['event_id', 'nfl_id', 'drill_name', 'rep', 't', 'dx', 'dy', 's_sg']].head()

,event_id,nfl_id,drill_name,rep,t,dx,dy,s_sg
118155,20240303100900_1709489409250_000,57126,FORTY_YARD_DASH,1,0.0,0.000000,0.000000,0.818730
118156,20240303100900_1709489409250_000,57126,FORTY_YARD_DASH,1,0.1,0.066667,0.000000,1.150952
118157,20240303100900_1709489409250_000,57126,FORTY_YARD_DASH,1,0.2,0.233333,-0.033333,1.580952
118158,20240303100900_1709489409250_000,57126,FORTY_YARD_DASH,1,0.3,0.433333,-0.033333,2.108730
118159,20240303100900_1709489409250_000,57126,FORTY_YARD_DASH,1,0.4,0.666667,-0.033333,2.757143


## Drill summary
One row per drill. `end_dx` and `end_dy` are where the median rep finishes relative to its start.

In [4]:
per_rep = reps.groupby(['drill_name', 'event_id']).agg(
    nfl_id=('nfl_id', 'first'), secs=('t', 'max'), peak_speed=('s_sg', 'max'),
    end_dx=('dx', 'last'), end_dy=('dy', 'last')).reset_index()

summary = per_rep.groupby('drill_name').agg(
    players=('nfl_id', 'nunique'), reps=('event_id', 'size'),
    secs=('secs', 'median'), peak_speed=('peak_speed', 'median'),
    end_dx=('end_dx', 'median'), end_dy=('end_dy', 'median'))
summary['reps_per_player'] = summary['reps'] / summary['players']
summary.round(2).sort_values('players', ascending=False)

,players,reps,secs,peak_speed,end_dx,end_dy,reps_per_player
drill_name,,,,,,,
PASS_PRO_MIRROR_DRILL,121,121,8.4,5.99,5.43,0.37,1.00
FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE,120,120,15.3,7.40,-15.97,-0.23,1.00
PASS_RUSH_DROP,120,239,4.3,6.07,-10.17,0.09,1.99
SCREEN_DRILL,119,120,6.6,6.99,15.45,-8.96,1.01
OL_PULL_DRILL_FOLD_BLOCK_RIGHT,119,119,3.7,7.08,10.49,3.23,1.00
OL_PULL_DRILL_DEEP_SHORT_PULL_LEFT,98,98,4.3,7.95,10.37,13.01,1.00
FORTY_YARD_DASH,95,179,5.5,9.35,40.47,-0.10,1.88
OL_PULL_DRILL_LONG_PULL_RIGHT,93,93,4.9,6.98,10.33,-8.57,1.00
SHORT_SHUTTLE,54,70,5.1,5.47,0.09,0.04,1.30


## Write the viewers
One file per drill. `marks` are the times circled on the path and drawn on the speed trace; change them per drill if a different cut matters.

In [ ]:
# any drill not listed gets the default (1.2, 1.5)
MARKS = {'PASS_RUSH_DROP': (1.2, 1.5)}

for drill in summary.index:
    path = write_viewer(reps, drill, OUT, marks=MARKS.get(drill, (1.2, 1.5)))
    print(path.name)

FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE.html
FORTY_YARD_DASH.html
OL_PULL_DRILL_DEEP_SHORT_PULL_LEFT.html
OL_PULL_DRILL_FOLD_BLOCK_RIGHT.html
OL_PULL_DRILL_LONG_PULL_RIGHT.html
PASS_PRO_MIRROR_DRILL.html
PASS_RUSH_DROP.html
SCREEN_DRILL.html
SHORT_SHUTTLE.html
THREE_CONE_DRILL.html


## Review notes
Fill in while going through the viewers.

| Drill | What it is | Start cue | Phases and timing | Sides / reps | Game counterpart | Use? |
|---|---|---|---|---|---|---|
| PASS_RUSH_DROP | | | | | | |
| PASS_PRO_MIRROR_DRILL | | | | | | |
| FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE | | | | | | |
| OL_PULL_DRILL_FOLD_BLOCK_RIGHT | | | | | | |
| OL_PULL_DRILL_DEEP_SHORT_PULL_LEFT | | | | | | |
| OL_PULL_DRILL_LONG_PULL_RIGHT | | | | | | |
| SCREEN_DRILL | | | | | | |
| SHORT_SHUTTLE | | | | | | |
| THREE_CONE_DRILL | | | | | | |
| FORTY_YARD_DASH | | | | | | |

## Decisions
Which drills and phases go forward, with the window for each. Copy into `decisions.md` when settled.

- PASS_RUSH_DROP: set phase 0 to 1.2 s (first pad dip median 1.6 s, 10th percentile 1.3 s). Two reps, one per side. Redirect phase under consideration.
- 